In [2]:
import datetime
from collections import namedtuple
import json
import urllib.request
from bs4 import BeautifulSoup
import logging
import random
import sys


MEALTIME_SWITCH = 14  # 14:00

Meal = namedtuple(
    "Meal",
    [
        "label",
        "price_student",
        "price_intern",
        "price_extern",
        "name",
        "description",
    ],
)

class Mensa:
    name = "Not available."
    alias = ""

    def get_meals(self):
        return []

class ETHMensa(Mensa):
    def __init__(self, name, alias, facility_id):
        self.name = name
        self.alias = alias
        self.facility_id = facility_id
        self.opening = ""
        self.closing = ""

    def get_meals(self):
        menus = []
        try:
            now = datetime.datetime.now()
            date = now.strftime("%Y-%m-%d")
            language = "en"  # "de" or "en"
            URL = f"https://idapps.ethz.ch/cookpit-pub-services/v1/weeklyrotas?client-id=ethz-wcms&lang={language}&rs-first=0&rs-size=50&valid-after={date}"

            with urllib.request.urlopen(URL) as request:
                meals = json.loads(request.read().decode())

            print(f"Got {len(meals['weekly-rota-array'])} facilities")

            facility = None
            for facility in meals["weekly-rota-array"]:
                valid_from = datetime.datetime.strptime(
                    facility["valid-from"], "%Y-%m-%d"
                )
                if not 0 <= (now - valid_from).days < 7:
                    continue
                if facility["facility-id"] != self.facility_id:
                    continue

                break
            else:
                return menus

            print(f"Found facility {facility['facility-id']}")

            day = None
            for day in facility["day-of-week-array"]:
                if "opening-hour-array" not in day.keys():
                    continue
                if len(day["opening-hour-array"][0]["meal-time-array"]) == 0:
                    continue
                if now.weekday() != day["day-of-week-code"]:
                    continue

                break
            else:
                return menus

            print(
                f"Found day {day['day-of-week-desc']} (code {day['day-of-week-code']}))"
            )

            meals = day["opening-hour-array"][0]["meal-time-array"]
            meal = meals[0]
            time_to = datetime.datetime.strptime(meal["time-to"], "%H:%M")
            if len(meals) > 1 and (
                now.hour == time_to.hour
                and now.minute > time_to.minute
                or now.hour > time_to.hour
            ):
                meal = meals[1]

            print(f"Found meal {meal['name']}")

            self.opening = meal["time-from"]
            self.closing = meal["time-to"]

            for m in meal["line-array"]:
                if len(m) == 1:
                    print(f"Found empty meal {m['name']}")

                    emoji_variants = ["🤷", "🤷‍♂️", "🤷‍♀️"]

                    menu = Meal(
                        label=m["name"],
                        price_student="$",
                        price_intern="$$",
                        price_extern="$$$",
                        name=random.choice(emoji_variants),
                        description="",
                    )

                    menus.append(menu)

                    continue

                print(f"Found meal {m['name']}")

                prices = [
                    (p["price"], p["customer-group-desc"])
                    for p in m["meal"]["meal-price-array"]
                ]
                student_price = next((p[0] for p in prices if "students" in p[1]), "$")
                intern_price = next((p[0] for p in prices if "internal" in p[1]), "$$")
                extern_price = next((p[0] for p in prices if "external" in p[1]), "$$$")

                menu = Meal(
                    label=m["name"],
                    price_student=student_price,
                    price_intern=intern_price,
                    price_extern=extern_price,
                    name=m["meal"]["name"],
                    description=m["meal"]["description"],
                )

                menus.append(menu)

            return menus
        except Exception as e:
            logging.error("Error while fetching ETH Mensa data")
            logging.error(e)
            return menus

In [3]:
clau = ETHMensa("Clausiusbar", "clausius", 3)
clau.get_meals()

Got 50 facilities
Found facility 3
Found day Thursday (code 4))
Found meal Lunch
Found meal WOK STREET
Found meal WOK GREEN
Found meal DELIGHT
Found meal GARDEN


[Meal(label='WOK STREET', price_student=13.2, price_intern=14.2, price_extern=16.5, name='KWANG PHANAENG', description='Chicken |\nPanang curry sauce (spicy) |\nCourgette | Carrots | Palatinate |\nJasmine rice\n\n'),
 Meal(label='WOK GREEN', price_student=12.9, price_intern=13.9, price_extern=15.9, name='PHANAENG', description='Planted Chicken |\nPanang Curry Sauce\nZucchini | Carrots | Palatinate |\nJasmine rice\n'),
 Meal(label='DELIGHT', price_student=9.2, price_intern=12.2, price_extern=14.5, name='GUA BAO', description='Pork | Coriander | Ednut butter sauce |\nPickled vegetables | Basmati rice |\n'),
 Meal(label='GARDEN', price_student=8.9, price_intern=11.9, price_extern=13.9, name='YAKISOBA', description='Fried noodles - Japan style\nChinese cabbage | onions | peppers | leek | onsen egg\n\n')]